In [1]:
import pandas as pd
import numpy as np

In [2]:
# =======================
# FILE PATHS
# =======================

airbnb_path = "AIR_BNB_Data.xlsx"
travel_path = "Kaggle_Data.xlsx"
eia_path = "eia_Data.xlsx"
tsa_path = "TSA_Traveler_Throughput_Data.xlsx"
bls_path = "BLS_CPI_Data.xlsx"

# =======================
# LOAD DATA
# =======================

airbnb = pd.read_excel(airbnb_path)
travel = pd.read_excel(travel_path)
eia = pd.read_excel(eia_path)
tsa = pd.read_excel(tsa_path)
bls = pd.read_excel(bls_path)

# =======================
# CLEAN COLUMN NAMES
# =======================

def clean_cols(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

airbnb = clean_cols(airbnb)
travel = clean_cols(travel)
eia = clean_cols(eia)
tsa = clean_cols(tsa)
bls = clean_cols(bls)

# =======================
# AIRBNB / LODGING
# =======================

airbnb["price"] = pd.to_numeric(airbnb["price"], errors = "coerce")

lodging_summary = (
    airbnb
    .groupby(["city", "room_type"], dropna=False)
    .agg(
        avg_nightly_price = ("price", "mean"),
        median_nighlty_price = ("price", "median"),
        listings = ("id", "count"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

# =======================
# KAGGLE TRAVEL COST DATA
# =======================

travel["duration_days"] = pd.to_numeric(travel["duration_days"], errors="coerce")
travel["accommodation_cost"] = pd.to_numeric(travel["accommodation_cost"], errors="coerce")
travel["transportation_cost"] = pd.to_numeric(travel["transportation_cost"], errors="coerce")

travel["total_known_trip_cost"] = (
    travel["accommodation_cost"] + travel["transportation_cost"]
)

travel_summary = (
    travel
    .groupby("destination", dropna=False)
    .agg(
        avg_duration = ("duration_days", "mean"),
        avg_accommodation_cost = ("accommodation_cost", "mean"),
        avg_transportation_cost = ("transportation_cost", "mean"),
        avg_total_known_trip_cost = ("total_known_trip_cost", "mean"),
        trips = ("trip_id", "count")
    )
    .reset_index()
)

# =======================
# EIA GAS DATA
# =======================

eia_date_col = eia.columns[0]
eia[eia_date_col] = pd.to_datetime(eia[eia_date_col], errors="coerce")

eia_long = eia.melt(
    id_vars=eia_date_col,
    var_name="source_key",
    value_name="gas_price"
)

eia_long["gas_price"] = pd.to_numeric(eia_long["gas_price"], errors="coerce")
eia_long["year"] = eia_long[eia_date_col].dt.year

gas_summary = (
    eia_long
    .groupby(["year", "source_key"], dropna=False)
    .agg(avg_gas_price=("gas_price", "mean"))
    .reset_index()
)

# =======================
# TSA TRAVELER DATA
# =======================

tsa["date"] = pd.to_datetime(tsa["date"], errors="coerce")
tsa["numbers"] = pd.to_numeric(tsa["numbers"], errors="coerce")
tsa["year"] = tsa["date"].dt.year

tsa_summary = (
    tsa
    .groupby("year")
    .agg(
        avg_daily_travelers=("numbers", "mean"),
        total_travelers=("numbers", "sum")
    )
    .reset_index()
)

# =======================
# BLS BPI Data
# =======================

month_cols = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

bls_long = bls.melt(
    id_vars=["series", "year"],
    value_vars=[c for c in month_cols if c in bls.columns],
    var_name="month",
    value_name="cpi_value"
)

bls_long["cpi_value"] = pd.to_numeric(bls_long["cpi_value"], errors="coerce")

bls_summary = (
    bls_long
    .groupby(["year", "series"], dropna=False)
    .agg(avg_cpi=("cpi_value", "mean"))
    .reset_index()
)

# =======================
# VACATION BUDGET SIMULATOR
# =======================

family_size = 4
vacation_days = 5

avg_daily_food_cost_per_person = 55
avg_activity_cost_per_person_per_day = 45

base_budget = travel_summary.copy()

base_budget["family_size"] = family_size
base_budget["vacation_days"] = vacation_days

base_budget["estimated_food_cost"] = (
    family_size * vacation_days * avg_daily_food_cost_per_person
)

base_budget["estimated_activity_cost"] = (
    family_size * vacation_days * avg_activity_cost_per_person_per_day
)

base_budget["estimated_total_vacation_cost"] = (
    base_budget["avg_accommodation_cost"]
    + base_budget["avg_transportation_cost"]
    + base_budget["estimated_food_cost"]
    + base_budget["estimated_activity_cost"]
)

base_budget["budget_level"] = pd.cut(
    base_budget["estimated_total_vacation_cost"],
    bins=[0, 2500, 5000, np.inf],
    labels=["Conservative", "Expected", "Comfortable"]
)

# =======================
# EXPORT CLEANED OUTPUTS
# =======================

lodging_summary.to_excel("lodge.xlsx", index=False)
travel_summary.to_excel("travel.xlsx", index=False)
gas_summary.to_excel("gas.xlsx", index=False)
tsa_summary.to_excel("air.xlsx", index=False)
bls_summary.to_excel("bls.xlsx", index=False)
base_budget.to_excel("vbs.xlsx", index=False)

print("Files created successfully.")

Files created successfully.


Building Streamlit

In [3]:
import plotly.express as px
import streamlit as st
from pathlib import Path
import pandas as pd

In [4]:
#PAGE CONFIGURATION

st.set_page_config(
    page_title="What Does a Family Vacation Cost in 2026?",
    layout="wide"
)

#FILE PATHS
BASE_DIR=Path.cwd()

#Data directionary
DATA_DIR=BASE_DIR

#Processed Data Files
budget_file = DATA_DIR / "vacation_budget_simulator.xlsx"
travel_file = DATA_DIR / "travel_summary.xlsx"
lodging_file = DATA_DIR / "lodging_summary.xlsx"
gas_file = DATA_DIR / "gas_summary.xlsx"
tsa_file = DATA_DIR / "tsa_summary.xlsx"
bls_file = DATA_DIR / "bls_summary.xlsx"


print(f"Working Directory: {BASE_DIR}")
print("All processed datasets loadded successfully")



Working Directory: c:\Users\pkfxb\OneDrive - Internal Revenue Service\Desktop\Personal\cwd\Summer_Travel\Export
All processed datasets loadded successfully


In [5]:
#LOAD DATA
budget_df = pd.read_excel(budget_file)
travel_df = pd.read_excel(travel_file)
lodging_df = pd.read_excel(lodging_file)
gas_df = pd.read_excel(gas_file)
tsa_df = pd.read_excel(tsa_file)
bls_df = pd.read_excel(bls_file)


# CLEAN COLUMN NAMES

def clean_cols(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

budget_df = clean_cols(budget_df)
travel_df = clean_cols(travel_df)
lodging_df = clean_cols(lodging_df)
gas_df = clean_cols(gas_df)
tsa_df = clean_cols(tsa_df)
bls_df = clean_cols(bls_df)

print("All processed datasets loaded successfully.")

All processed datasets loaded successfully.


In [6]:
budget_df.columns.tolist()

['destination',
 'avg_duration',
 'avg_accommodation_cost',
 'avg_transportation_cost',
 'avg_total_known_trip_cost',
 'trips',
 'family_size',
 'vacation_days',
 'estimated_food_cost',
 'estimated_activity_cost',
 'estimated_total_vacation_cost',
 'budget_level']

#Building website

In [11]:
from pathlib import Path 

st.set_page_config(
    page_title="What Does a Family Vacation Cost in 2026?",
    layout="wide"
)

#FILE PATHS
BASE_DIR = Path.cwd()

budget_file = BASE_DIR / "vacation_budget_simulator.xlsx"
gas_file = BASE_DIR / "gas_summary.xlsx"

#Load Data
def load_data():
    budget = pd.read_excel(budget_file)
    gas = pd.read_excel(gas_file)

    return budget, gas

budget_df, gas_df = load_data()

#Basic Data Prep

numeric_columns = [
    "avg_duration",
    "avg_accommodation_cost",
    "avg_transportation_cost",
    "avg_total_known_trip_cost",
    "family_size",
    "vacation_days",
    "estimated_food_cost",
    "estimated_activity_cost",
    "estimated_total_vacation_cost",
]

for column in numeric_columns:
    budget_df[column] = pd.to_numeric(
        budget_df[column],
        errors="coerce",
    )

gas_df["avg_gas_price"] = pd.to_numeric(
    gas_df["avg_gas_price"],
    errors="coerce",
)

#Page Header

st.title("What Does a Family Vacation Cost in 2026?")

st.write(
    """
    An interactive Clarity with Data project exploring the cost of 
    lodging, transporation, food, activities, and fuel.
    """
)

#Project Summary

average_vacation_cost = (
    budget_df["estimated_total_vacation_cost"].mean()
)

average_lodging_cost = (
    budget_df["avg_accommodation_cost"].mean()
)

average_transportation_cost = (
    budget_df["avg_transportation_cost"].mean()
)

average_gas_price = (
    gas_df["avg_gas_price"].mean()
)

metric_1, metric_2, metric_3, metric_4, = st.columns(4)

metric_1.metric(
    "Average Vacation Cost",
    f"${average_vacation_cost:,.0f}",
)

metric_2.metric(
    "Average Lodging Cost",
    f"${average_lodging_cost:,.0f}",
)

metric_3.metric(
    "Average Transportation Cost",
    f"${average_transportation_cost:,.0f}"
)

metric_4.metric(
    "Average Gas Price",
    f"${average_gas_price:,.2f}"
)

#Vacation Calculator

st.divider()

st.header("Vacation Budget Calculator")

destinations = sorted(
    budget_df["destination"]
    .dropna()
    .astype(str)
    .unique()
)

control_1, control_2, control_3 = st.columns(3)

selected_destination = control_1.selectbox(
    "Destination",
    destinations,
)

family_size = control_2.slider(
    "Family size",
    min_value=1,
    max_value=10,
    value=4,
    step=1,
)

vacation_days = control_3.slider(
    "Vacation length",
    min_value=2,
    max_value=14,
    value=5,
    step=1,
)

control_4, control_5 = st.columns(2)

daily_food_cost = control_4.slider(
    "Daily food cost per person",
    min_value=20,
    max_value=150,
    value=55,
    step=5,
)

daily_activity_cost = control_5.slider(
    "Daily activity cost per person",
    min_value=0,
    max_value=150,
    value=45,
    step=5,
)

#Calculate Selected Budget

selected_row = budget_df[
    budget_df["destination"].astype(str)
    == selected_destination
].iloc[0]

source_duration = selected_row["avg_duration"]

if pd.isna(source_duration) or source_duration <= 0:
    source_duration = selected_row["vacation_days"]

if pd.isna(source_duration) or source_duration <= 0:
    source_duration = 5

lodging_cost = (
    selected_row["avg_accommodation_cost"]
    *vacation_days
    / source_duration
)

transportation_cost = (
    selected_row["avg_transportation_cost"]
)

food_cost = (
    family_size
    *vacation_days
    *daily_food_cost
)

activity_cost = (
    family_size
    *vacation_days
    *daily_activity_cost
)

estimated_total = (
    lodging_cost
    +transportation_cost
    +food_cost
    +activity_cost
)

planning_buffer = estimated_total * 0.10
comfortable_budget = estimated_total + planning_buffer

#Calculator results
result_1, result_2, result_3 = st.columns(3)

result_1.metric(
    "Base Estimate",
    f"${estimated_total:,.0f}",
)

result_2.metric(
    "10% Planning Buffer",
    f"${planning_buffer:,.0f}",
)

result_3.metric(
    "Comfortable Budget",
    f"${comfortable_budget:,.0f}",
)

#Cost Breakdown Chart

cost_breakdown = pd.DataFrame(
    {
        "Category": [
            "Lodging",
            "Transportation",
            "Food",
            "Activities"
        ],
        "Estimated Cost":[
            lodging_cost,
            transportation_cost,
            food_cost,
            activity_cost
        ],
    }
)

chart = px.bar(
    cost_breakdown,
    x="Category",
    y="Estimated Cost",
    title=f"Estimated Cost Break: {selected_destination}",
    text="Estimated Cost"
)

chart.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
)

chart.update_layout(
    yaxis_tickprefix="$",
    showlegend=False,
)

st.plotly_chart(
    chart,
    use_container_width=True,
)

#Footer

st.divider()

st.caption(
    "Clarity With Data | Where Insight Meets Intention"
)

DeltaGenerator()